# 01 ? Baseline: Reprodu??o da Fase 1

Este notebook reproduz o pipeline da Fase 1 usando os m?dulos refatorados em `src/`.

**Objetivo:** confirmar as m?tricas de linha de base antes de aplicar o Algoritmo Gen?tico.

| Modelo | Accuracy | Precision | Recall | F1 |
|--------|----------|-----------|--------|----|
| Regress?o Log?stica | ~83% | ~13% | **~44%** | ~0.21 |
| Random Forest | ~93% | ~26% | ~24% | ~0.25 |

> **Recall** ? a m?trica principal (contexto m?dico: minimizar falsos negativos).

## 0. Setup ? instalar depend?ncias e importar m?dulos

In [ ]:
import sys
import os

# Ensure src/ is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.preprocessing import prepare_pipeline
from src.models import (
    train_logistic_regression,
    train_random_forest,
    evaluate_model,
    save_model,
    save_metrics,
)

print('Imports OK')

## 1. Carregar e pr?-processar os dados

Executa o pipeline completo: `load_data ? clean_data ? encode_features ? split_and_balance (SMOTE)`.

> Se o CSV n?o existir, rode primeiro: `python data/download_data.py`

In [ ]:
X_train, X_test, y_train, y_test = prepare_pipeline()

print(f'\nFeatures: {list(X_train.columns)}')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')

## 2. An?lise de correla??o

Verifica??o visual das features mais correlacionadas com `stroke`.

In [ ]:
import numpy as np

# Reconstruct full df (pre-split) just for correlation
from src.preprocessing import load_data, clean_data, encode_features
df_full = encode_features(clean_data(load_data()))

corr = df_full.corr(numeric_only=True)
stroke_corr = corr['stroke'].drop('stroke').sort_values(ascending=False)

plt.figure(figsize=(8, 5))
stroke_corr.plot(kind='barh', color=['#e74c3c' if v > 0 else '#3498db' for v in stroke_corr])
plt.title('Correlacao com stroke (target)')
plt.xlabel('Pearson correlation')
plt.tight_layout()
plt.show()

print('Top 5 features correlacionadas com stroke:')
print(stroke_corr.head())

## 3. Treinar modelos baseline

In [ ]:
print('Treinando Logistic Regression (baseline)...')
lr_model = train_logistic_regression(X_train, y_train)

print('\nTreinando Random Forest (baseline)...')
rf_model = train_random_forest(X_train, y_train)

## 4. Avaliar modelos e confirmar m?tricas da Fase 1

In [ ]:
lr_metrics = evaluate_model(lr_model, X_test, y_test, 'Logistic Regression (baseline)')
rf_metrics = evaluate_model(rf_model, X_test, y_test, 'Random Forest (baseline)')

## 5. Visualiza??o comparativa

In [ ]:
metrics_names = ['accuracy', 'precision', 'recall', 'f1']
lr_vals = [lr_metrics[m] for m in metrics_names]
rf_vals = [rf_metrics[m] for m in metrics_names]

x = range(len(metrics_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar([i - width/2 for i in x], lr_vals, width, label='Logistic Regression', color='#3498db')
bars2 = ax.bar([i + width/2 for i in x], rf_vals, width, label='Random Forest', color='#e74c3c')

ax.set_ylabel('Score')
ax.set_title('Baseline: Logistic Regression vs Random Forest')
ax.set_xticks(list(x))
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1'])
ax.legend()
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}',
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 6. Salvar modelos e m?tricas

Os arquivos `.joblib` em `results/` s?o usados pelo **Integrante 2** (LLM integration)
e pelo **Integrante 1** nos experimentos do Algoritmo Gen?tico.

In [ ]:
save_model(lr_model, 'logistic_regression')
save_model(rf_model, 'random_forest')

save_metrics([lr_metrics, rf_metrics], 'baseline_metrics.json')

print('\nBaseline concluido. Modelos salvos em results/')

## 7. Conclus?o

- **Regress?o Log?stica** tem o melhor Recall (~44%) ? m?trica principal para diagn?stico de AVC.
- **Random Forest** tem melhor Accuracy e Precision, mas Recall mais baixo (~24%).
- O Algoritmo Gen?tico (Fase 2) buscar? melhorar o Recall mantendo F1 razo?vel.

**Pr?ximo passo:** `notebooks/02_genetic_algorithm.ipynb`